# Marginal Distributions

A joint distribution answers questions about *pairs* — but sometimes you suddenly stop caring about one of the variables. If you have the full distribution of ages **and** heights but only want heights, you *aggregate over all the ages*: the result is the **marginal distribution**. It reduces a two-dimensional distribution back to one dimension — summing columns (or rows) of the joint table in the discrete case, flattening the density mountain along one axis in the continuous case. It is based on the fourth video of Lesson 4 (Probability Distributions with Multiple Variables).

## 1. Introduction

Two important concepts extract one-variable information from a joint distribution:

- the **marginal distribution** — *ignore* one variable entirely by aggregating over it (this notebook);
- the **conditional distribution** — take a *slice* at one fixed value, e.g. the heights of only the 30-year-olds (next video).

Marginals are the everyday tool for summarizing the behavior of one variable at a time when your data is inherently multivariate.

After completing this notebook you will be able to:

- Compute marginal PMFs as **row sums and column sums** of a joint table, and use the formula $P_Y(y_j) = \sum_i P_{XY}(x_i, y_j)$.
- Recover the original single-variable PMFs from the joint (dice), and derive new ones (the sum's triangle).
- Extend the idea to continuous joints: marginals as integrals that flatten the density mountain along one axis.
- Explain what information survives marginalization — and what is lost.

## 2. Theory

### Aggregating out a variable

Recall the 10-children joint distribution:

| $P_{XY}$ | 45 | 46 | 47 | 48 | 49 | 50 |
|---|---|---|---|---|---|---|
| **7** | 0.1 | 0.2 | 0 | 0 | 0 | 0 |
| **8** | 0 | 0 | 0.2 | 0 | 0 | 0 |
| **9** | 0 | 0 | 0 | 0 | 0.3 | 0.1 |
| **10** | 0 | 0 | 0 | 0 | 0 | 0.1 |

Suppose we no longer care about age — we only want the distribution of **height**. For each height value, add up all the probabilities in its **column**:

$$
P_Y(45) = 0.1, \quad P_Y(46) = 0.2, \quad P_Y(47) = 0.2, \quad P_Y(48) = 0, \quad P_Y(49) = 0.3, \quad P_Y(50) = 0.1 + 0.1 = 0.2
$$

This is the **marginal distribution of height** — a distribution of one variable that completely ignores the other. The general formula:

$$
P_Y(y_j) = \sum_i P_{XY}(x_i, y_j) \qquad \text{(sum down the column of } y_j\text{)}
$$

Quiz from the video: $P_Y(50) = \sum_i P_{XY}(x_i, 50) = 0.1 + 0.1 = \tfrac{2}{10}$. ✓

Symmetrically, if we only care about **age**, we add over every **row**:

$$
P_X(x_i) = \sum_j P_{XY}(x_i, y_j)
\qquad\Rightarrow\qquad
P_X(7) = 0.3,\ P_X(8) = 0.2,\ P_X(9) = 0.4,\ P_X(10) = 0.1
$$

(The name "marginal" comes from exactly this picture: the sums written in the *margins* of the table.)

### Marginals reduce dimensions

A marginal takes a higher-dimensional distribution and produces one of fewer dimensions: our 2-variable joint becomes a 1-variable PMF. With a scatter/heat-grid of 50 children, the age marginal is obtained by summing the grid across the vertical axis and the height marginal by summing across the horizontal axis.

### Two dice: marginals recover what we started with

For $X$ = first die, $Y$ = second die (the flat $\tfrac1{36}$ table): summing any row gives $6 \cdot \tfrac1{36} = \tfrac16$, and the same for columns — the marginals are exactly the original single-die PMFs. Nothing new, and that is the point: the joint *contains* the individual distributions.

### First die and the sum: a new marginal appears

For the (first die, sum) band table, aggregating over the rows gives the distribution of the **sum**: $\tfrac1{36}, \tfrac2{36}, \dots, \tfrac6{36}, \dots, \tfrac1{36}$ — the triangle peaking at 7, because many more pairs add to 7 than to 2 or 12. Quiz: $P_Y(10) = \tfrac{3}{36}$ (the sum of the 10th row). We went *to* two variables and *back* to one.

### Continuous marginals: flattening the mountain

For the call-center density (waiting time × rating), the same operation uses integrals instead of sums:

$$
f_X(x) = \int f_{XY}(x, y)\,dy \qquad f_Y(y) = \int f_{XY}(x, y)\,dx
$$

Aggregate the density along one axis and the 2D mountain collapses into a 1D curve — the histogram you would see if you only ever recorded that one variable.

## 3. Worked Examples

### Example 1: both marginals of the children table

Column sums give the height marginal $(0.1, 0.2, 0.2, 0, 0.3, 0.2)$; row sums give the age marginal $(0.3, 0.2, 0.4, 0.1)$. Each sums to 1 — a marginal is a genuine PMF.

### Example 2: the video's two quizzes

$$
P_Y(50) = \sum_i P_{XY}(x_i, 50) = 0 + 0 + 0.1 + 0.1 = \tfrac{2}{10}
\qquad\qquad
P_X(7) = \sum_j P_{XY}(7, y_j) = 0.1 + 0.2 = \tfrac{3}{10}
$$

### Example 3: the sum's marginal

Row 10 of the (die, sum) table contains $\tfrac1{36}$ at $x = 4, 5, 6$ (pairs $4{+}6, 5{+}5, 6{+}4$):

$$
P_Y(10) = \tfrac{3}{36}
$$

— matching the triangle histogram we counted directly in Lesson 2.

### Example 4: what marginals lose

The children's marginals $(0.3, 0.2, 0.4, 0.1)$ and $(0.1, 0.2, 0.2, 0, 0.3, 0.2)$ do **not** determine the joint: the same two marginals are consistent with many different tables (e.g. the independence product table, which has no zeros at all). Aggregation is a one-way street — the dependence information stays behind.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

ages = [7, 8, 9, 10]
heights = [45, 46, 47, 48, 49, 50]
joint = pd.DataFrame(
    [[0.1, 0.2, 0.0, 0.0, 0.0, 0.0],
     [0.0, 0.0, 0.2, 0.0, 0.0, 0.0],
     [0.0, 0.0, 0.0, 0.0, 0.3, 0.1],
     [0.0, 0.0, 0.0, 0.0, 0.0, 0.1]],
    index=pd.Index(ages, name="age"),
    columns=pd.Index(heights, name="height"))
print(joint)

### Marginals are one line of pandas: sum over the axis you ignore

In [ ]:
marginal_height = joint.sum(axis=0)      # sum down each column
marginal_age = joint.sum(axis=1)         # sum along each row

print("marginal distribution of height (column sums):")
print(marginal_height)
print(f"\nmarginal distribution of age (row sums):")
print(marginal_age)
print(f"\nboth are valid PMFs: sums = {marginal_height.sum()}, {marginal_age.sum()}")
print(f"\nquiz answers:  P_Y(50) = {marginal_height[50]:.1f},  P_X(7) = {marginal_age[7]:.1f}")

### The dice examples: recovering old PMFs and building new ones

In [ ]:
faces = np.arange(1, 7)

# flat joint of two independent dice
flat = pd.DataFrame(1 / 36, index=faces, columns=faces)
print("marginal of the first die (row sums of the flat table):")
print(flat.sum(axis=1).round(4))

# (first die, sum) joint
sums = np.arange(2, 13)
die_sum = pd.DataFrame(0.0, index=sums, columns=faces)
for d1 in faces:
    for d2 in faces:
        die_sum.loc[d1 + d2, d1] += 1 / 36

marginal_sum = die_sum.sum(axis=1)
print("\nmarginal of the sum (row sums of the band table), in 36ths:")
print((marginal_sum * 36).round(0).astype(int))
print(f"\nquiz: P_Y(10) = {marginal_sum[10]:.4f} = 3/36 = {3/36:.4f}")

### Continuous marginals: integrate the density along one axis

Rebuild the call-center joint density on a grid and flatten it each way; compare with the histograms of each variable recorded alone — they match, because that is exactly what a marginal is.

In [ ]:
rng = np.random.default_rng(42)

n = 200_000
quick = rng.random(n) < 0.5
wait = np.clip(np.where(quick, rng.normal(2.4, 1.6, n), rng.normal(7.6, 1.6, n)), 0, 10)
rate = np.clip(np.where(quick, rng.normal(8.0, 1.7, n), rng.normal(2.6, 1.7, n)), 0, 10)

density, xe, ye = np.histogram2d(wait, rate, bins=50, range=[[0, 10], [0, 10]],
                                 density=True)
dx, dy = xe[1] - xe[0], ye[1] - ye[0]

f_wait = density.sum(axis=1) * dy        # integrate out the rating
f_rate = density.sum(axis=0) * dx        # integrate out the waiting time

print(f"area under marginal of waiting time: {(f_wait * dx).sum():.4f}")
print(f"area under marginal of rating:       {(f_rate * dy).sum():.4f}")
print("Flattening the 2D density each way yields two genuine 1D densities.")

## 5. Visualizations

### The classic picture: joint in the middle, marginals in the margins

In [ ]:
fig = plt.figure(figsize=(9, 7))
gs = fig.add_gridspec(2, 2, width_ratios=[4, 1.2], height_ratios=[1.2, 4],
                      hspace=0.08, wspace=0.08)
ax_joint = fig.add_subplot(gs[1, 0])
ax_top = fig.add_subplot(gs[0, 0], sharex=ax_joint)
ax_right = fig.add_subplot(gs[1, 1], sharey=ax_joint)

im = ax_joint.imshow(joint.values, cmap="Blues", aspect="auto",
                     extent=[44.5, 50.5, 10.5, 6.5])
for i, a in enumerate(ages):
    for j, h in enumerate(heights):
        v = joint.values[i, j]
        if v > 0:
            ax_joint.text(h, a, f"{v:.1f}", ha="center", va="center",
                          color="white" if v > 0.2 else "black")
ax_joint.set_xlabel("height y (inches)")
ax_joint.set_ylabel("age x (years)")
ax_joint.set_xticks(heights)
ax_joint.set_yticks(ages)

ax_top.bar(heights, marginal_height.values, color="tab:orange", edgecolor="black")
ax_top.set_ylabel("$P_Y$")
ax_top.set_title("marginal of height = column sums (top),  marginal of age = row sums (right)")
plt.setp(ax_top.get_xticklabels(), visible=False)

ax_right.barh(ages, marginal_age.values, color="tab:green", edgecolor="black")
ax_right.set_xlabel("$P_X$")
plt.setp(ax_right.get_yticklabels(), visible=False)

plt.show()

### The 50-children heat grid with its two marginals

An expanded dataset: darker cells hold more children; summing the grid vertically gives the age marginal, horizontally the height marginal.

In [ ]:
rng = np.random.default_rng(7)

n_kids = 50
kid_age = rng.integers(7, 12, size=n_kids)
kid_height = np.round(38 + 2.1 * kid_age + rng.normal(0, 1.6, n_kids)).astype(int)

grid = pd.crosstab(kid_age, kid_height)
fig = plt.figure(figsize=(9.5, 7))
gs = fig.add_gridspec(2, 2, width_ratios=[4, 1.2], height_ratios=[1.2, 4],
                      hspace=0.08, wspace=0.08)
ax_joint = fig.add_subplot(gs[1, 0])
ax_top = fig.add_subplot(gs[0, 0])
ax_right = fig.add_subplot(gs[1, 1])

hs = grid.columns.values
as_ = grid.index.values
im = ax_joint.imshow(grid.values, cmap="Blues", aspect="auto",
                     extent=[hs.min() - 0.5, hs.max() + 0.5,
                             as_.max() + 0.5, as_.min() - 0.5])
ax_joint.set_xlabel("height (inches)")
ax_joint.set_ylabel("age (years)")

ax_top.bar(hs, grid.sum(axis=0).values, color="tab:orange", edgecolor="black")
ax_top.set_xlim(hs.min() - 0.5, hs.max() + 0.5)
ax_top.set_ylabel("count")
ax_top.set_title("50 children: joint heat grid and its two marginals")

ax_right.barh(as_, grid.sum(axis=1).values, color="tab:green", edgecolor="black")
ax_right.set_ylim(as_.max() + 0.5, as_.min() - 0.5)
ax_right.set_xlabel("count")

plt.show()

### Flattening the call-center mountain

The 2D density with its two continuous marginals: each 1D curve is the histogram you'd get recording only that variable.

In [ ]:
xc = (xe[:-1] + xe[1:]) / 2
yc = (ye[:-1] + ye[1:]) / 2

fig = plt.figure(figsize=(9.5, 7.5))
gs = fig.add_gridspec(2, 2, width_ratios=[4, 1.2], height_ratios=[1.2, 4],
                      hspace=0.08, wspace=0.08)
ax_joint = fig.add_subplot(gs[1, 0])
ax_top = fig.add_subplot(gs[0, 0], sharex=ax_joint)
ax_right = fig.add_subplot(gs[1, 1], sharey=ax_joint)

ax_joint.imshow(density.T, origin="lower", extent=[0, 10, 0, 10],
                cmap="viridis", aspect="auto")
ax_joint.set_xlabel("waiting time (minutes)")
ax_joint.set_ylabel("rating")

ax_top.plot(xc, f_wait, color="tab:blue", linewidth=2)
ax_top.fill_between(xc, f_wait, alpha=0.3, color="tab:blue")
ax_top.set_ylabel("$f_X$")
ax_top.set_title("call center: joint density and its two marginals")
plt.setp(ax_top.get_xticklabels(), visible=False)

ax_right.plot(f_rate, yc, color="tab:red", linewidth=2)
ax_right.fill_betweenx(yc, f_rate, alpha=0.3, color="tab:red")
ax_right.set_xlabel("$f_Y$")
plt.setp(ax_right.get_yticklabels(), visible=False)

plt.show()

## 6. Exercises

### Basic

**Exercise 1.** For the children joint table, verify by hand and by code that both marginals sum to 1, and compute $P_X(9)$ and $P_Y(47)$ with the summation formula.

**Exercise 2.** For the (first die, sum) table, compute $P_Y(4)$ and $P_Y(11)$ from row sums, and name the dice pairs behind each.

### Intermediate

**Exercise 3.** Build the joint table of $X$ = first die and $M$ = maximum of two dice (from the previous notebook), and compute the marginal of $M$. Check it against the direct formula $P(M = m) = \frac{2m - 1}{36}$.

**Exercise 4.** Marginals of a *uniform* joint on the unit square: for $f_{XY} = 1$ on $[0,1]^2$, show numerically that both marginals are the uniform density 1 on $[0,1]$. Then repeat for $f_{XY}(x,y) = 4xy$ — what are its marginals, and are $X$ and $Y$ independent?

### Advanced

**Exercise 5.** Two different joints, same marginals: take the children's marginals and build the **independence product table** $P_X(x)P_Y(y)$. Confirm its marginals equal the originals, yet the table differs from the true joint (show the largest cell difference). What real-world conclusion would someone wrongly draw from the product table?

**Exercise 6.** Marginalizing a 3-variable joint: for three fair dice, the joint of $(X_1, X_2, X_3)$ is uniform $\tfrac{1}{216}$. Marginalize out $X_3$ (sum over its axis) and confirm you get the flat $\tfrac1{36}$ table of $(X_1, X_2)$; then marginalize again to reach $P_{X_1}$. Dimension reduction works one variable at a time.

## 7. Solutions

In [ ]:
# Exercise 1: sums and two lookups
print(f"Exercise 1: height marginal sums to {joint.sum(axis=0).sum()}, "
      f"age marginal sums to {joint.sum(axis=1).sum()}")
print(f"P_X(9)  = sum of row 9  = {joint.loc[9].sum():.1f}")
print(f"P_Y(47) = sum of col 47 = {joint[47].sum():.1f}")

In [ ]:
# Exercise 2: two rows of the (die, sum) table
print(f"Exercise 2: P_Y(4)  = {die_sum.loc[4].sum():.4f} = 3/36  (pairs 1+3, 2+2, 3+1)")
print(f"            P_Y(11) = {die_sum.loc[11].sum():.4f} = 2/36  (pairs 5+6, 6+5)")

In [ ]:
# Exercise 3: marginal of the maximum
die_max = pd.DataFrame(0.0, index=faces, columns=faces)
for a in faces:
    for b in faces:
        die_max.loc[max(a, b), a] += 1 / 36

marginal_max = die_max.sum(axis=1)
print("Exercise 3: marginal of M (in 36ths) vs formula (2m-1)/36:")
for m in faces:
    print(f"  M = {m}: {marginal_max[m] * 36:.0f}/36   formula: {2 * m - 1}/36")

In [ ]:
# Exercise 4: continuous marginals by numerical integration
x = np.linspace(0, 1, 501)
y = np.linspace(0, 1, 501)
Xg, Yg = np.meshgrid(x, y, indexing="ij")

for name, f in [("uniform f = 1", np.ones_like(Xg)), ("f = 4xy", 4 * Xg * Yg)]:
    fx = np.trapezoid(f, y, axis=1)          # integrate out y
    print(f"{name}: marginal f_X at x = 0.25, 0.5, 0.75 -> "
          f"{fx[125]:.3f}, {fx[250]:.3f}, {fx[375]:.3f}")
print()
print("Uniform: marginals are flat 1 -> uniform on [0,1].")
print("f = 4xy: f_X(x) = 2x (and f_Y(y) = 2y). Since 2x * 2y = 4xy = f_XY, the")
print("joint factors into its marginals -- X and Y are independent here.")

In [ ]:
# Exercise 5: same marginals, different joint
product = pd.DataFrame(np.outer(marginal_age, marginal_height),
                       index=joint.index, columns=joint.columns)
print("Exercise 5: independence product table:")
print(product.round(3))
print(f"\nmarginals preserved? rows: {np.allclose(product.sum(axis=1), marginal_age)}, "
      f"cols: {np.allclose(product.sum(axis=0), marginal_height)}")
gap = (product - joint).abs()
cell = gap.stack().idxmax()
print(f"largest difference at (age, height) = {cell}: "
      f"product {product.loc[cell]:.2f} vs true {joint.loc[cell]:.2f}")
print("From the product table one would conclude that a 7-year-old is as likely")
print("to be 50 inches as anyone -- the true joint says age and height move")
print("together. Marginals alone cannot restore that lost dependence.")

In [ ]:
# Exercise 6: marginalizing a 3D joint step by step
joint3 = np.full((6, 6, 6), 1 / 216)
two_dice = joint3.sum(axis=2)                     # marginalize out X3
one_die = two_dice.sum(axis=1)                    # then X2
print(f"Exercise 6: after summing out X3: all cells 1/36? "
      f"{np.allclose(two_dice, 1/36)}")
print(f"after also summing out X2: P(X1) = {one_die.round(4)} (1/6 each)")
print("Each marginalization removes one dimension; chaining them walks a joint")
print("distribution down to any subset of its variables.")

## 8. Conclusion

- The **marginal distribution** is what remains when you aggregate a joint distribution over the variables you no longer care about: $P_Y(y_j) = \sum_i P_{XY}(x_i, y_j)$ — **column sums** for one variable, **row sums** ($P_X(x_i) = \sum_j P_{XY}(x_i, y_j)$) for the other. The name comes from writing these sums in the table's margins.
- Children's table: height marginal $(0.1, 0.2, 0.2, 0, 0.3, 0.2)$, age marginal $(0.3, 0.2, 0.4, 0.1)$; quizzes $P_Y(50) = \tfrac2{10}$, $P_X(7) = \tfrac3{10}$, and for the dice band $P_Y(10) = \tfrac3{36}$.
- Marginals **reduce dimension** (2 variables → 1) and recover familiar objects: the flat dice joint returns the $\tfrac16$ PMFs; the (die, sum) joint yields the sum's triangle.
- Continuous version: replace sums with integrals, $f_X(x) = \int f_{XY}(x, y)\,dy$ — flattening the density mountain along one axis.
- Marginals summarize but **lose the dependence**: many joints share the same marginals, so you cannot rebuild a joint from its marginals alone.

Next: the other extraction — **conditional distributions**, the *slices* of a joint.